# Exoplanet Discovery Analysis
**Data Source:** NASA Exoplanet Archive (TAP API)  
**Tools:** Python · Pandas · Matplotlib · Seaborn  

Analyzing how exoplanet discovery grew from rare announcements to a catalog of 6,000+ confirmed worlds — driven by transit detection and survey missions like Kepler and TESS.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import requests
import io

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

In [ ]:
# Query NASA Exoplanet Archive via TAP API
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

query = """
SELECT pl_name, disc_year, discoverymethod, disc_facility,
       pl_rade, pl_orbper, pl_bmasse
FROM pscomppars
WHERE disc_year IS NOT NULL
"""

params = {"query": query, "format": "csv"}
response = requests.get(TAP_URL, params=params)
df = pd.read_csv(io.StringIO(response.text))

print(f"Total confirmed exoplanets: {len(df):,}")
print(f"Discovery years: {df['disc_year'].min():.0f} – {df['disc_year'].max():.0f}")
df.head()

In [ ]:
# 1. Cumulative discoveries over time
yearly = df.groupby('disc_year').size().reset_index(name='count')
yearly['cumulative'] = yearly['count'].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(yearly['disc_year'], yearly['count'], color='steelblue', alpha=0.8)
axes[0].set_title('Exoplanet Discoveries Per Year', fontsize=14)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Planets Discovered')

axes[1].plot(yearly['disc_year'], yearly['cumulative'], color='darkorange', linewidth=2.5)
axes[1].fill_between(yearly['disc_year'], yearly['cumulative'], alpha=0.15, color='darkorange')
axes[1].set_title('Cumulative Confirmed Exoplanets', fontsize=14)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Planets')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../data/discovery_trends.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2. Discovery methods over time
method_year = df.groupby(['disc_year', 'discoverymethod']).size().unstack(fill_value=0)
top_methods = df['discoverymethod'].value_counts().head(4).index.tolist()

fig, ax = plt.subplots(figsize=(12, 5))
method_year[top_methods].plot(kind='bar', stacked=True, ax=ax,
                               colormap='tab10', width=0.85)
ax.set_title('Planets Discovered Per Year by Method', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.legend(title='Method', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/discovery_by_method.png', bbox_inches='tight')
plt.show()

# Method share
print("\nDiscovery method share:")
print(df['discoverymethod'].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# 3. Top discovery facilities
top_facilities = df['disc_facility'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_facilities.sort_values().plot(kind='barh', ax=ax, color='teal', alpha=0.85)
ax.set_title('Top 10 Discovery Facilities', fontsize=14)
ax.set_xlabel('Confirmed Planets')
plt.tight_layout()
plt.savefig('../data/top_facilities.png', bbox_inches='tight')
plt.show()

In [ ]:
# 4. Planet radius vs orbital period — the diversity of exoplanets
plot_df = df.dropna(subset=['pl_rade', 'pl_orbper'])
plot_df = plot_df[(plot_df['pl_rade'] > 0) & (plot_df['pl_orbper'] > 0)]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    plot_df['pl_orbper'], plot_df['pl_rade'],
    alpha=0.3, s=12, c='steelblue'
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Planet Radius vs Orbital Period', fontsize=14)
ax.set_xlabel('Orbital Period (days, log scale)')
ax.set_ylabel('Planet Radius (Earth radii, log scale)')

# Mark Earth reference
ax.scatter([365.25], [1.0], color='green', s=80, zorder=5, label='Earth')
ax.legend()
plt.tight_layout()
plt.savefig('../data/radius_vs_period.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5. Summary statistics
print("=== Summary ===")
print(f"Total confirmed exoplanets: {len(df):,}")
print(f"Transit share: {(df['discoverymethod']=='Transit').mean()*100:.1f}%")

kepler_tess = df[df['disc_facility'].str.contains('Kepler|TESS|K2', na=False, case=False)]
print(f"Kepler + TESS + K2 share: {len(kepler_tess)/len(df)*100:.1f}%")
print(f"Discoveries since 2009: {(df['disc_year']>=2009).mean()*100:.1f}%")